In [1]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2013 or earlier for training and samples ending in 2014 for testing.

"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
from aeon.classification.convolution_based import MultiRocketHydraClassifier
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2013  
test_mask = forecast_end_years == 2014   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

/opt/anaconda3/lib/python3.12/site-packages/aeon/base/__init__.py:24: FutureWarning: The aeon package will soon be releasing v1.0.0 with the removal of legacy modules and interfaces such as BaseTransformer and BaseForecaster. This will contain breaking changes. See aeon-toolkit.org for more information. Set aeon.AEON_DEPRECATION_WARNING or the AEON_DEPRECATION_WARNING environmental variable to 'False' to disable this warning.
  warnings.warn(


Training shapes: (10921, 30, 5) (10921, 7, 5) (10921,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [2]:
"""
This code selects the best available compute device for PyTorch model training
and evaluation.

It first checks whether Apple Silicon MPS acceleration is available. If not, it
checks for a CUDA-compatible NVIDIA GPU. If neither hardware accelerator is
available, it falls back to using the CPU.

"""
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [3]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 38.22it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9044    0.9138    0.9091       290
           1     0.6528    0.6267    0.6395        75

    accuracy                         0.8548       365
   macro avg     0.7786    0.7702    0.7743       365
weighted avg     0.8527    0.8548    0.8537       365

AUROC (seed 0): 0.9133
AUPRC (seed 0): 0.7023
AUPRC class 0 (seed 0): 0.9773

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 38.18it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9366    0.9172    0.9268       290
           1     0.7037    0.7600    0.7308        75

    accuracy                         0.8849       365
   macro avg     0.8202    0.8386    0.8288       365
weighted avg     0.8888    0.8849    0.8865       365

AUROC (seed 1): 0.9190
AUPRC (seed 1): 0.7489
AUPRC class 0 (seed 1): 0.9779

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.9122    0.9310    0.9215       290
           1     0.7101    0.6533    0.6806        75

    accuracy                         0.8740       365
   macro avg     0.8112    0.7922    0.8010       365
weighted avg     0.8707    0.8740    0.8720       365

AUROC (seed 2): 0.9313
AUPRC (seed 2): 0.7742
AUPRC class 0 (seed 2): 0.9818

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 37.54it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9110    0.8828    0.8967       290
           1     0.5952    0.6667    0.6289        75

    accuracy                         0.8384       365
   macro avg     0.7531    0.7747    0.7628       365
weighted avg     0.8461    0.8384    0.8417       365

AUROC (seed 3): 0.8863
AUPRC (seed 3): 0.6480
AUPRC class 0 (seed 3): 0.9693

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 35.87it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8984    0.9448    0.9210       290
           1     0.7333    0.5867    0.6519        75

    accuracy                         0.8712       365
   macro avg     0.8158    0.7657    0.7864       365
weighted avg     0.8645    0.8712    0.8657       365

AUROC (seed 4): 0.9331
AUPRC (seed 4): 0.7783
AUPRC class 0 (seed 4): 0.9824

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9133  AUPRC=0.7023AUPRC0=0.9773
Seed 1: AUROC=0.9190  AUPRC=0.7489AUPRC0=0.9779
Seed 2: AUROC=0.9313  AUPRC=0.7742AUPRC0=0.9818
Seed 3: AUROC=0.8863  AUPRC=0.6480AUPRC0=0.9693
Seed 4: AUROC=0.9331  AUPRC=0.7783AUPRC0=0.9824


In [4]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 22/22 [00:00<00:00, 29.14it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 36.75it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9399    0.9172    0.9284       290
           1     0.7073    0.7733    0.7389        75

    accuracy                         0.8877       365
   macro avg     0.8236    0.8453    0.8337       365
weighted avg     0.8921    0.8877    0.8895       365

AUROC (seed 0): 0.9421
AUPRC (seed 0): 0.8202
AUPRC class 0 (seed 0): 0.9847

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9424    0.9034    0.9225       290
           1     0.6782    0.7867    0.7284        75

    accuracy                         0.8795       365
   macro avg     0.8103    0.8451    0.8255       365
weighted avg     0.8881    0.8795    0.8826       365

AUROC (seed 3): 0.9318
AUPRC (seed 3): 0.7909
AUPRC class 0 (seed 3): 0.9821

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 22/22 [00:00<00:00, 27.96it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 22/22 [00:00<00:00, 36.52it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9082    0.9207    0.9144       290
           1     0.6761    0.6400    0.6575        75

    accuracy                         0.8630       365
   macro avg     0.7921    0.7803    0.7860       365
weighted avg     0.8605    0.8630    0.8616       365

AUROC (seed 4): 0.9067
AUPRC (seed 4): 0.7021
AUPRC class 0 (seed 4): 0.9745

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9421  AUPRC=0.8202AUPRC0=0.9847
Seed 1: AUROC=0.9285  AUPRC=0.7879AUPRC0=0.9807
Seed 2: AUROC=0.8813  AUPRC=0.6367AUPRC0=0.9675
Seed 3: AUROC=0.9318  AUPRC=0.7909AUPRC0=0.9821
Seed 4: AUROC=0.9067  AUPRC=0.7021AUPRC0=0.9745


In [5]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 22/22 [00:04<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.9199    0.9103    0.9151       290
           1     0.6667    0.6933    0.6797        75

    accuracy                         0.8658       365
   macro avg     0.7933    0.8018    0.7974       365
weighted avg     0.8678    0.8658    0.8667       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8996
AUPRC (MC mean over 100 samples, seed 0) : 0.6946
AUPRC class 0 (seed 0): 0.9724

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 22/22 [00:04<00:00,  5.30it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.9049    0.9517    0.9277       290
           1     0.7667    0.6133    0.6815        75

    accuracy                         0.8822       365
   macro avg     0.8358    0.7825    0.8046       365
weighted avg     0.8765    0.8822    0.8771       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9380
AUPRC (MC mean over 100 samples, seed 1) : 0.7889
AUPRC class 0 (seed 1): 0.9834

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9218    0.9345    0.9281       290
           1     0.7324    0.6933    0.7123        75

    accuracy                         0.8849       365
   macro avg     0.8271    0.8139    0.8202       365
weighted avg     0.8829    0.8849    0.8837       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9275
AUPRC (MC mean over 100 samples, seed 2) : 0.7780
AUPRC class 0 (seed 2): 0.9800

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 22/22 [00:04<00:00,  5.29it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8935    0.9552    0.9233       290
           1     0.7636    0.5600    0.6462        75

    accuracy                         0.8740       365
   macro avg     0.8286    0.7576    0.7847       365
weighted avg     0.8669    0.8740    0.8664       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9251
AUPRC (MC mean over 100 samples, seed 3) : 0.7553
AUPRC class 0 (seed 3): 0.9802

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 22/22 [00:04<00:00,  5.29it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.9085    0.9241    0.9162       290
           1     0.6857    0.6400    0.6621        75

    accuracy                         0.8658       365
   macro avg     0.7971    0.7821    0.7892       365
weighted avg     0.8627    0.8658    0.8640       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9094
AUPRC (MC mean over 100 samples, seed 4) : 0.7283
AUPRC class 0 (seed 4): 0.9747

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8996  AUPRC=0.6946AUPRC0=0.9724
Seed 1: AUROC=0.9380  AUPRC=0.7889AUPRC0=0.9834
Seed 2: AUROC=0.9275  AUPRC=0.7780AUPRC0=0.9800
Seed 3: AUROC=0.9251  AUPRC=0.75

In [3]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8736    0.8345    0.8536       290
           1     0.4545    0.5333    0.4908        75

    accuracy                         0.7726       365
   macro avg     0.6641    0.6839    0.6722       365
weighted avg     0.7875    0.7726    0.7791       365

AUROC (seed 0): 0.6839
AUPRC (seed 0): 0.3383
AUPRC class 0 (seed 0): 0.8605

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8779    0.9172    0.8971       290
           1     0.6129    0.5067    0.5547        75

    accuracy                         0.8329       365
   macro avg     0.7454    0.7120    0.7259       365
weighted avg     0.8234    0.8329    0.8268       365

AUROC (seed 1): 0.7120
AUPRC (seed 1): 0.4119
AUPRC class 0 (seed 1): 0.8710

Training MultiRocketHydraClassifi

In [4]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2014 or earlier for training and samples ending in 2015 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2014  
test_mask = forecast_end_years == 2015   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (11286, 30, 5) (11286, 7, 5) (11286,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [5]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8435    0.8800    0.8613       300
           1     0.3077    0.2462    0.2735        65

    accuracy                         0.7671       365
   macro avg     0.5756    0.5631    0.5674       365
weighted avg     0.7480    0.7671    0.7567       365

AUROC (seed 0): 0.5631
AUPRC (seed 0): 0.2100
AUPRC class 0 (seed 0): 0.8409

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8436    0.8633    0.8534       300
           1     0.2931    0.2615    0.2764        65

    accuracy                         0.7562       365
   macro avg     0.5684    0.5624    0.5649       365
weighted avg     0.7456    0.7562    0.7506       365

AUROC (seed 1): 0.5624
AUPRC (seed 1): 0.2082
AUPRC class 0 (seed 1): 0.8407

Training MultiRocketHydraClassifi

In [7]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.70it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9212    0.8967    0.9088       300
           1     0.5753    0.6462    0.6087        65

    accuracy                         0.8521       365
   macro avg     0.7483    0.7714    0.7587       365
weighted avg     0.8596    0.8521    0.8553       365

AUROC (seed 0): 0.8447
AUPRC (seed 0): 0.5752
AUPRC class 0 (seed 0): 0.9583

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.72it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9292    0.7433    0.8259       300
           1     0.3840    0.7385    0.5053        65

    accuracy                         0.7425       365
   macro avg     0.6566    0.7409    0.6656       365
weighted avg     0.8321    0.7425    0.7688       365

AUROC (seed 1): 0.8122
AUPRC (seed 1): 0.4859
AUPRC class 0 (seed 1): 0.9503

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.9147    0.8933    0.9039       300
           1     0.5556    0.6154    0.5839        65

    accuracy                         0.8438       365
   macro avg     0.7351    0.7544    0.7439       365
weighted avg     0.8507    0.8438    0.8469       365

AUROC (seed 2): 0.8701
AUPRC (seed 2): 0.6576
AUPRC class 0 (seed 2): 0.9661

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.68it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9030    0.9000    0.9015       300
           1     0.5455    0.5538    0.5496        65

    accuracy                         0.8384       365
   macro avg     0.7242    0.7269    0.7256       365
weighted avg     0.8393    0.8384    0.8388       365

AUROC (seed 3): 0.8298
AUPRC (seed 3): 0.5813
AUPRC class 0 (seed 3): 0.9536

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.63it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9058    0.9300    0.9178       300
           1     0.6316    0.5538    0.5902        65

    accuracy                         0.8630       365
   macro avg     0.7687    0.7419    0.7540       365
weighted avg     0.8570    0.8630    0.8594       365

AUROC (seed 4): 0.8727
AUPRC (seed 4): 0.6685
AUPRC class 0 (seed 4): 0.9676

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8447  AUPRC=0.5752AUPRC0=0.9583
Seed 1: AUROC=0.8122  AUPRC=0.4859AUPRC0=0.9503
Seed 2: AUROC=0.8701  AUPRC=0.6576AUPRC0=0.9661
Seed 3: AUROC=0.8298  AUPRC=0.5813AUPRC0=0.9536
Seed 4: AUROC=0.8727  AUPRC=0.6685AUPRC0=0.9676


In [8]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 23/23 [00:00<00:00, 28.48it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 36.95it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9024    0.8933    0.8978       300
           1     0.5294    0.5538    0.5414        65

    accuracy                         0.8329       365
   macro avg     0.7159    0.7236    0.7196       365
weighted avg     0.8359    0.8329    0.8343       365

AUROC (seed 0): 0.8300
AUPRC (seed 0): 0.6002
AUPRC class 0 (seed 0): 0.9510

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9082    0.9233    0.9157       300
           1     0.6167    0.5692    0.5920        65

    accuracy                         0.8603       365
   macro avg     0.7624    0.7463    0.7539       365
weighted avg     0.8563    0.8603    0.8581       365

AUROC (seed 3): 0.8659
AUPRC (seed 3): 0.6697
AUPRC class 0 (seed 3): 0.9636

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 23/23 [00:00<00:00, 27.58it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 31.32it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9118    0.9300    0.9208       300
           1     0.6441    0.5846    0.6129        65

    accuracy                         0.8685       365
   macro avg     0.7779    0.7573    0.7668       365
weighted avg     0.8641    0.8685    0.8660       365

AUROC (seed 4): 0.8730
AUPRC (seed 4): 0.6927
AUPRC class 0 (seed 4): 0.9671

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8300  AUPRC=0.6002AUPRC0=0.9510
Seed 1: AUROC=0.8606  AUPRC=0.6695AUPRC0=0.9629
Seed 2: AUROC=0.8449  AUPRC=0.6396AUPRC0=0.9564
Seed 3: AUROC=0.8659  AUPRC=0.6697AUPRC0=0.9636
Seed 4: AUROC=0.8730  AUPRC=0.6927AUPRC0=0.9671


In [9]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.25it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.9071    0.9433    0.9248       300
           1     0.6792    0.5538    0.6102        65

    accuracy                         0.8740       365
   macro avg     0.7931    0.7486    0.7675       365
weighted avg     0.8665    0.8740    0.8688       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8748
AUPRC (MC mean over 100 samples, seed 0) : 0.7155
AUPRC class 0 (seed 0): 0.9665

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.14it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8832    0.9833    0.9306       300
           1     0.8387    0.4000    0.5417        65

    accuracy                         0.8795       365
   macro avg     0.8610    0.6917    0.7361       365
weighted avg     0.8753    0.8795    0.8613       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8705
AUPRC (MC mean over 100 samples, seed 1) : 0.7104
AUPRC class 0 (seed 1): 0.9596

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9082    0.9233    0.9157       300
           1     0.6167    0.5692    0.5920        65

    accuracy                         0.8603       365
   macro avg     0.7624    0.7463    0.7539       365
weighted avg     0.8563    0.8603    0.8581       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.8651
AUPRC (MC mean over 100 samples, seed 2) : 0.6455
AUPRC class 0 (seed 2): 0.9619

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.20it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.9061    0.9333    0.9195       300
           1     0.6429    0.5538    0.5950        65

    accuracy                         0.8658       365
   macro avg     0.7745    0.7436    0.7573       365
weighted avg     0.8593    0.8658    0.8618       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.8655
AUPRC (MC mean over 100 samples, seed 3) : 0.6759
AUPRC class 0 (seed 3): 0.9621

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.24it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8981    0.9700    0.9327       300
           1     0.7805    0.4923    0.6038        65

    accuracy                         0.8849       365
   macro avg     0.8393    0.7312    0.7682       365
weighted avg     0.8772    0.8849    0.8741       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8670
AUPRC (MC mean over 100 samples, seed 4) : 0.6693
AUPRC class 0 (seed 4): 0.9628

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8748  AUPRC=0.7155AUPRC0=0.9665
Seed 1: AUROC=0.8705  AUPRC=0.7104AUPRC0=0.9596
Seed 2: AUROC=0.8651  AUPRC=0.6455AUPRC0=0.9619
Seed 3: AUROC=0.8655  AUPRC=0.67

In [6]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2015 or earlier for training and samples ending in 2016 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2015  
test_mask = forecast_end_years == 2016   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (11651, 30, 5) (11651, 7, 5) (11651,)
Test shapes: (366, 30, 5) (366, 7, 5) (366,)


In [7]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8633    0.8421    0.8526       285
           1     0.4886    0.5309    0.5089        81

    accuracy                         0.7732       366
   macro avg     0.6760    0.6865    0.6807       366
weighted avg     0.7804    0.7732    0.7765       366

AUROC (seed 0): 0.6865
AUPRC (seed 0): 0.3632
AUPRC class 0 (seed 0): 0.8499

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8837    0.8000    0.8398       285
           1     0.4722    0.6296    0.5397        81

    accuracy                         0.7623       366
   macro avg     0.6780    0.7148    0.6897       366
weighted avg     0.7927    0.7623    0.7734       366

AUROC (seed 1): 0.7148
AUPRC (seed 1): 0.3793
AUPRC class 0 (seed 1): 0.8627

Training MultiRocketHydraClassifi

In [11]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.32it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9217    0.9088    0.9152       285
           1     0.6941    0.7284    0.7108        81

    accuracy                         0.8689       366
   macro avg     0.8079    0.8186    0.8130       366
weighted avg     0.8713    0.8689    0.8700       366

AUROC (seed 0): 0.9258
AUPRC (seed 0): 0.7940
AUPRC class 0 (seed 0): 0.9769

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.06it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9110    0.9333    0.9220       285
           1     0.7432    0.6790    0.7097        81

    accuracy                         0.8770       366
   macro avg     0.8271    0.8062    0.8158       366
weighted avg     0.8738    0.8770    0.8750       366

AUROC (seed 1): 0.9350
AUPRC (seed 1): 0.7870
AUPRC class 0 (seed 1): 0.9819

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.9257    0.8737    0.8989       285
           1     0.6289    0.7531    0.6854        81

    accuracy                         0.8470       366
   macro avg     0.7773    0.8134    0.7922       366
weighted avg     0.8600    0.8470    0.8517       366

AUROC (seed 2): 0.9193
AUPRC (seed 2): 0.7588
AUPRC class 0 (seed 2): 0.9766

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.27it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8923    0.9298    0.9107       285
           1     0.7101    0.6049    0.6533        81

    accuracy                         0.8579       366
   macro avg     0.8012    0.7674    0.7820       366
weighted avg     0.8520    0.8579    0.8537       366

AUROC (seed 3): 0.9021
AUPRC (seed 3): 0.7359
AUPRC class 0 (seed 3): 0.9699

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.29it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9377    0.8982    0.9176       285
           1     0.6882    0.7901    0.7356        81

    accuracy                         0.8743       366
   macro avg     0.8130    0.8442    0.8266       366
weighted avg     0.8825    0.8743    0.8773       366

AUROC (seed 4): 0.9364
AUPRC (seed 4): 0.7870
AUPRC class 0 (seed 4): 0.9819

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9258  AUPRC=0.7940AUPRC0=0.9769
Seed 1: AUROC=0.9350  AUPRC=0.7870AUPRC0=0.9819
Seed 2: AUROC=0.9193  AUPRC=0.7588AUPRC0=0.9766
Seed 3: AUROC=0.9021  AUPRC=0.7359AUPRC0=0.9699
Seed 4: AUROC=0.9364  AUPRC=0.7870AUPRC0=0.9819


In [12]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 23/23 [00:00<00:00, 28.20it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 36.71it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9294    0.8316    0.8778       285
           1     0.5676    0.7778    0.6562        81

    accuracy                         0.8197       366
   macro avg     0.7485    0.8047    0.7670       366
weighted avg     0.8493    0.8197    0.8288       366

AUROC (seed 0): 0.8955
AUPRC (seed 0): 0.6880
AUPRC class 0 (seed 0): 0.9701

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8951    0.8982    0.8967       285
           1     0.6375    0.6296    0.6335        81

    accuracy                         0.8388       366
   macro avg     0.7663    0.7639    0.7651       366
weighted avg     0.8381    0.8388    0.8384       366

AUROC (seed 3): 0.8900
AUPRC (seed 3): 0.7199
AUPRC class 0 (seed 3): 0.9660

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 23/23 [00:00<00:00, 28.47it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 23/23 [00:00<00:00, 37.02it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9492    0.8526    0.8983       285
           1     0.6182    0.8395    0.7120        81

    accuracy                         0.8497       366
   macro avg     0.7837    0.8461    0.8052       366
weighted avg     0.8760    0.8497    0.8571       366

AUROC (seed 4): 0.9163
AUPRC (seed 4): 0.7487
AUPRC class 0 (seed 4): 0.9754

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8955  AUPRC=0.6880AUPRC0=0.9701
Seed 1: AUROC=0.9208  AUPRC=0.7148AUPRC0=0.9783
Seed 2: AUROC=0.9124  AUPRC=0.7730AUPRC0=0.9731
Seed 3: AUROC=0.8900  AUPRC=0.7199AUPRC0=0.9660
Seed 4: AUROC=0.9163  AUPRC=0.7487AUPRC0=0.9754


In [13]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.19it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.9258    0.9193    0.9225       285
           1     0.7229    0.7407    0.7317        81

    accuracy                         0.8798       366
   macro avg     0.8243    0.8300    0.8271       366
weighted avg     0.8809    0.8798    0.8803       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.9432
AUPRC (MC mean over 100 samples, seed 0) : 0.8404
AUPRC class 0 (seed 0): 0.9836

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.21it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.9007    0.9544    0.9267       285
           1     0.7969    0.6296    0.7034        81

    accuracy                         0.8825       366
   macro avg     0.8488    0.7920    0.8151       366
weighted avg     0.8777    0.8825    0.8773       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9497
AUPRC (MC mean over 100 samples, seed 1) : 0.8515
AUPRC class 0 (seed 1): 0.9857

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9175    0.9368    0.9271       285
           1     0.7600    0.7037    0.7308        81

    accuracy                         0.8852       366
   macro avg     0.8388    0.8203    0.8289       366
weighted avg     0.8827    0.8852    0.8836       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9371
AUPRC (MC mean over 100 samples, seed 2) : 0.8154
AUPRC class 0 (seed 2): 0.9820

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.19it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.9091    0.9474    0.9278       285
           1     0.7826    0.6667    0.7200        81

    accuracy                         0.8852       366
   macro avg     0.8458    0.8070    0.8239       366
weighted avg     0.8811    0.8852    0.8818       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9434
AUPRC (MC mean over 100 samples, seed 3) : 0.8434
AUPRC class 0 (seed 3): 0.9835

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 23/23 [00:04<00:00,  5.20it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.9119    0.9439    0.9276       285
           1     0.7746    0.6790    0.7237        81

    accuracy                         0.8852       366
   macro avg     0.8433    0.8114    0.8256       366
weighted avg     0.8815    0.8852    0.8825       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9469
AUPRC (MC mean over 100 samples, seed 4) : 0.8425
AUPRC class 0 (seed 4): 0.9850

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.9432  AUPRC=0.8404AUPRC0=0.9836
Seed 1: AUROC=0.9497  AUPRC=0.8515AUPRC0=0.9857
Seed 2: AUROC=0.9371  AUPRC=0.8154AUPRC0=0.9820
Seed 3: AUROC=0.9434  AUPRC=0.84

In [8]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2016 or earlier for training and samples ending in 2017 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2016  
test_mask = forecast_end_years == 2017   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (12017, 30, 5) (12017, 7, 5) (12017,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [9]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.7931    0.9068    0.8462       279
           1     0.4348    0.2326    0.3030        86

    accuracy                         0.7479       365
   macro avg     0.6139    0.5697    0.5746       365
weighted avg     0.7087    0.7479    0.7182       365

AUROC (seed 0): 0.5697
AUPRC (seed 0): 0.2819
AUPRC class 0 (seed 0): 0.7904

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8167    0.8781    0.8463       279
           1     0.4769    0.3605    0.4106        86

    accuracy                         0.7562       365
   macro avg     0.6468    0.6193    0.6284       365
weighted avg     0.7366    0.7562    0.7436       365

AUROC (seed 1): 0.6193
AUPRC (seed 1): 0.3226
AUPRC class 0 (seed 1): 0.8103

Training MultiRocketHydraClassifi

In [15]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.93it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9078    0.9176    0.9127       279
           1     0.7229    0.6977    0.7101        86

    accuracy                         0.8658       365
   macro avg     0.8153    0.8076    0.8114       365
weighted avg     0.8642    0.8658    0.8649       365

AUROC (seed 0): 0.8870
AUPRC (seed 0): 0.7468
AUPRC class 0 (seed 0): 0.9552

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.94it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9351    0.8781    0.9057       279
           1     0.6699    0.8023    0.7302        86

    accuracy                         0.8603       365
   macro avg     0.8025    0.8402    0.8179       365
weighted avg     0.8726    0.8603    0.8644       365

AUROC (seed 1): 0.9011
AUPRC (seed 1): 0.7430
AUPRC class 0 (seed 1): 0.9666

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8815    0.9068    0.8940       279
           1     0.6667    0.6047    0.6341        86

    accuracy                         0.8356       365
   macro avg     0.7741    0.7557    0.7641       365
weighted avg     0.8309    0.8356    0.8328       365

AUROC (seed 2): 0.8740
AUPRC (seed 2): 0.7082
AUPRC class 0 (seed 2): 0.9566

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.96it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9091    0.8244    0.8647       279
           1     0.5625    0.7326    0.6364        86

    accuracy                         0.8027       365
   macro avg     0.7358    0.7785    0.7505       365
weighted avg     0.8274    0.8027    0.8109       365

AUROC (seed 3): 0.8433
AUPRC (seed 3): 0.5805
AUPRC class 0 (seed 3): 0.9433

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.92it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9300    0.8566    0.8918       279
           1     0.6296    0.7907    0.7010        86

    accuracy                         0.8411       365
   macro avg     0.7798    0.8237    0.7964       365
weighted avg     0.8592    0.8411    0.8468       365

AUROC (seed 4): 0.9070
AUPRC (seed 4): 0.7845
AUPRC class 0 (seed 4): 0.9679

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8870  AUPRC=0.7468AUPRC0=0.9552
Seed 1: AUROC=0.9011  AUPRC=0.7430AUPRC0=0.9666
Seed 2: AUROC=0.8740  AUPRC=0.7082AUPRC0=0.9566
Seed 3: AUROC=0.8433  AUPRC=0.5805AUPRC0=0.9433
Seed 4: AUROC=0.9070  AUPRC=0.7845AUPRC0=0.9679


In [16]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 24/24 [00:00<00:00, 28.88it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.31it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8803    0.8961    0.8881       279
           1     0.6420    0.6047    0.6228        86

    accuracy                         0.8274       365
   macro avg     0.7611    0.7504    0.7554       365
weighted avg     0.8241    0.8274    0.8256       365

AUROC (seed 0): 0.8703
AUPRC (seed 0): 0.6662
AUPRC class 0 (seed 0): 0.9568

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9234    0.8638    0.8926       279
           1     0.6346    0.7674    0.6947        86

    accuracy                         0.8411       365
   macro avg     0.7790    0.8156    0.7937       365
weighted avg     0.8553    0.8411    0.8460       365

AUROC (seed 3): 0.9166
AUPRC (seed 3): 0.8022
AUPRC class 0 (seed 3): 0.9731

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 24/24 [00:00<00:00, 28.92it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 24/24 [00:00<00:00, 37.28it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9028    0.9319    0.9171       279
           1     0.7532    0.6744    0.7117        86

    accuracy                         0.8712       365
   macro avg     0.8280    0.8032    0.8144       365
weighted avg     0.8675    0.8712    0.8687       365

AUROC (seed 4): 0.9109
AUPRC (seed 4): 0.7734
AUPRC class 0 (seed 4): 0.9703

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8703  AUPRC=0.6662AUPRC0=0.9568
Seed 1: AUROC=0.9075  AUPRC=0.7909AUPRC0=0.9684
Seed 2: AUROC=0.8976  AUPRC=0.7133AUPRC0=0.9679
Seed 3: AUROC=0.9166  AUPRC=0.8022AUPRC0=0.9731
Seed 4: AUROC=0.9109  AUPRC=0.7734AUPRC0=0.9703


In [17]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 24/24 [00:04<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8767    0.9176    0.8967       279
           1     0.6849    0.5814    0.6289        86

    accuracy                         0.8384       365
   macro avg     0.7808    0.7495    0.7628       365
weighted avg     0.8315    0.8384    0.8336       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.9181
AUPRC (MC mean over 100 samples, seed 0) : 0.7756
AUPRC class 0 (seed 0): 0.9747

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 24/24 [00:04<00:00,  5.25it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.9197    0.9032    0.9114       279
           1     0.7033    0.7442    0.7232        86

    accuracy                         0.8658       365
   macro avg     0.8115    0.8237    0.8173       365
weighted avg     0.8687    0.8658    0.8670       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9433
AUPRC (MC mean over 100 samples, seed 1) : 0.8454
AUPRC class 0 (seed 1): 0.9828

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9000    0.9032    0.9016       279
           1     0.6824    0.6744    0.6784        86

    accuracy                         0.8493       365
   macro avg     0.7912    0.7888    0.7900       365
weighted avg     0.8487    0.8493    0.8490       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9270
AUPRC (MC mean over 100 samples, seed 2) : 0.7875
AUPRC class 0 (seed 2): 0.9779

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 24/24 [00:04<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8885    0.9140    0.9011       279
           1     0.6923    0.6279    0.6585        86

    accuracy                         0.8466       365
   macro avg     0.7904    0.7709    0.7798       365
weighted avg     0.8423    0.8466    0.8439       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9190
AUPRC (MC mean over 100 samples, seed 3) : 0.7671
AUPRC class 0 (seed 3): 0.9752

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 24/24 [00:04<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.9097    0.9032    0.9065       279
           1     0.6932    0.7093    0.7011        86

    accuracy                         0.8575       365
   macro avg     0.8015    0.8063    0.8038       365
weighted avg     0.8587    0.8575    0.8581       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9064
AUPRC (MC mean over 100 samples, seed 4) : 0.7227
AUPRC class 0 (seed 4): 0.9710

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.9181  AUPRC=0.7756AUPRC0=0.9747
Seed 1: AUROC=0.9433  AUPRC=0.8454AUPRC0=0.9828
Seed 2: AUROC=0.9270  AUPRC=0.7875AUPRC0=0.9779
Seed 3: AUROC=0.9190  AUPRC=0.76

In [10]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2017 or earlier for training and samples ending in 2018 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2017  
test_mask = forecast_end_years == 2018   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (12382, 30, 5) (12382, 7, 5) (12382,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [11]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.7436    0.7546    0.7491       269
           1     0.2826    0.2708    0.2766        96

    accuracy                         0.6274       365
   macro avg     0.5131    0.5127    0.5128       365
weighted avg     0.6223    0.6274    0.6248       365

AUROC (seed 0): 0.5127
AUPRC (seed 0): 0.2683
AUPRC class 0 (seed 0): 0.7420

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.7639    0.8662    0.8118       269
           1     0.4000    0.2500    0.3077        96

    accuracy                         0.7041       365
   macro avg     0.5820    0.5581    0.5598       365
weighted avg     0.6682    0.7041    0.6792       365

AUROC (seed 1): 0.5581
AUPRC (seed 1): 0.2973
AUPRC class 0 (seed 1): 0.7603

Training MultiRocketHydraClassifi

In [19]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.70it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8963    0.8030    0.8471       269
           1     0.5726    0.7396    0.6455        96

    accuracy                         0.7863       365
   macro avg     0.7344    0.7713    0.7463       365
weighted avg     0.8111    0.7863    0.7940       365

AUROC (seed 0): 0.8297
AUPRC (seed 0): 0.6681
AUPRC class 0 (seed 0): 0.9172

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.81it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8952    0.8253    0.8588       269
           1     0.5983    0.7292    0.6573        96

    accuracy                         0.8000       365
   macro avg     0.7467    0.7772    0.7580       365
weighted avg     0.8171    0.8000    0.8058       365

AUROC (seed 1): 0.8290
AUPRC (seed 1): 0.7209
AUPRC class 0 (seed 1): 0.9071

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8839    0.8773    0.8806       269
           1     0.6633    0.6771    0.6701        96

    accuracy                         0.8247       365
   macro avg     0.7736    0.7772    0.7754       365
weighted avg     0.8259    0.8247    0.8252       365

AUROC (seed 2): 0.8132
AUPRC (seed 2): 0.6903
AUPRC class 0 (seed 2): 0.8950

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.77it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8556    0.8810    0.8681       269
           1     0.6364    0.5833    0.6087        96

    accuracy                         0.8027       365
   macro avg     0.7460    0.7322    0.7384       365
weighted avg     0.7979    0.8027    0.7999       365

AUROC (seed 3): 0.8158
AUPRC (seed 3): 0.7052
AUPRC class 0 (seed 3): 0.9043

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.91it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8856    0.7770    0.8277       269
           1     0.5349    0.7188    0.6133        96

    accuracy                         0.7616       365
   macro avg     0.7102    0.7479    0.7205       365
weighted avg     0.7934    0.7616    0.7713       365

AUROC (seed 4): 0.8349
AUPRC (seed 4): 0.7040
AUPRC class 0 (seed 4): 0.9248

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8297  AUPRC=0.6681AUPRC0=0.9172
Seed 1: AUROC=0.8290  AUPRC=0.7209AUPRC0=0.9071
Seed 2: AUROC=0.8132  AUPRC=0.6903AUPRC0=0.8950
Seed 3: AUROC=0.8158  AUPRC=0.7052AUPRC0=0.9043
Seed 4: AUROC=0.8349  AUPRC=0.7040AUPRC0=0.9248


In [20]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 25/25 [00:00<00:00, 28.90it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.34it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8893    0.8364    0.8621       269
           1     0.6071    0.7083    0.6538        96

    accuracy                         0.8027       365
   macro avg     0.7482    0.7724    0.7580       365
weighted avg     0.8151    0.8027    0.8073       365

AUROC (seed 0): 0.8283
AUPRC (seed 0): 0.7248
AUPRC class 0 (seed 0): 0.9023

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8915    0.8550    0.8729       269
           1     0.6355    0.7083    0.6700        96

    accuracy                         0.8164       365
   macro avg     0.7635    0.7817    0.7714       365
weighted avg     0.8242    0.8164    0.8195       365

AUROC (seed 3): 0.8333
AUPRC (seed 3): 0.7003
AUPRC class 0 (seed 3): 0.9200

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 25/25 [00:00<00:00, 28.84it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.21it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8938    0.7509    0.8162       269
           1     0.5180    0.7500    0.6128        96

    accuracy                         0.7507       365
   macro avg     0.7059    0.7505    0.7145       365
weighted avg     0.7950    0.7507    0.7627       365

AUROC (seed 4): 0.8195
AUPRC (seed 4): 0.6981
AUPRC class 0 (seed 4): 0.9118

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8283  AUPRC=0.7248AUPRC0=0.9023
Seed 1: AUROC=0.8420  AUPRC=0.7544AUPRC0=0.9119
Seed 2: AUROC=0.8420  AUPRC=0.7600AUPRC0=0.9210
Seed 3: AUROC=0.8333  AUPRC=0.7003AUPRC0=0.9200
Seed 4: AUROC=0.8195  AUPRC=0.6981AUPRC0=0.9118


In [21]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.24it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8885    0.8587    0.8733       269
           1     0.6381    0.6979    0.6667        96

    accuracy                         0.8164       365
   macro avg     0.7633    0.7783    0.7700       365
weighted avg     0.8226    0.8164    0.8190       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8678
AUPRC (MC mean over 100 samples, seed 0) : 0.8019
AUPRC class 0 (seed 0): 0.9404

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.27it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8906    0.8476    0.8686       269
           1     0.6239    0.7083    0.6634        96

    accuracy                         0.8110       365
   macro avg     0.7572    0.7780    0.7660       365
weighted avg     0.8205    0.8110    0.8146       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8660
AUPRC (MC mean over 100 samples, seed 1) : 0.7979
AUPRC class 0 (seed 1): 0.9388

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8893    0.8959    0.8926       269
           1     0.7021    0.6875    0.6947        96

    accuracy                         0.8411       365
   macro avg     0.7957    0.7917    0.7937       365
weighted avg     0.8401    0.8411    0.8406       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.8664
AUPRC (MC mean over 100 samples, seed 2) : 0.7985
AUPRC class 0 (seed 2): 0.9374

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.27it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8838    0.7918    0.8353       269
           1     0.5484    0.7083    0.6182        96

    accuracy                         0.7699       365
   macro avg     0.7161    0.7501    0.7267       365
weighted avg     0.7956    0.7699    0.7782       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.8618
AUPRC (MC mean over 100 samples, seed 3) : 0.7924
AUPRC class 0 (seed 3): 0.9341

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.27it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8714    0.9071    0.8889       269
           1     0.7059    0.6250    0.6630        96

    accuracy                         0.8329       365
   macro avg     0.7887    0.7660    0.7759       365
weighted avg     0.8279    0.8329    0.8295       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8568
AUPRC (MC mean over 100 samples, seed 4) : 0.7815
AUPRC class 0 (seed 4): 0.9265

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8678  AUPRC=0.8019AUPRC0=0.9404
Seed 1: AUROC=0.8660  AUPRC=0.7979AUPRC0=0.9388
Seed 2: AUROC=0.8664  AUPRC=0.7985AUPRC0=0.9374
Seed 3: AUROC=0.8618  AUPRC=0.79

In [12]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2018 or earlier for training and samples ending in 2019 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2018  
test_mask = forecast_end_years == 2019   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (12747, 30, 5) (12747, 7, 5) (12747,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [13]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8901    0.8294    0.8587       293
           1     0.4565    0.5833    0.5122        72

    accuracy                         0.7808       365
   macro avg     0.6733    0.7063    0.6854       365
weighted avg     0.8046    0.7808    0.7903       365

AUROC (seed 0): 0.7063
AUPRC (seed 0): 0.3485
AUPRC class 0 (seed 0): 0.8752

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8610    0.8669    0.8639       293
           1     0.4429    0.4306    0.4366        72

    accuracy                         0.7808       365
   macro avg     0.6519    0.6487    0.6503       365
weighted avg     0.7785    0.7808    0.7797       365

AUROC (seed 1): 0.6487
AUPRC (seed 1): 0.3030
AUPRC class 0 (seed 1): 0.8533

Training MultiRocketHydraClassifi

In [23]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.65it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9140    0.8703    0.8916       293
           1     0.5581    0.6667    0.6076        72

    accuracy                         0.8301       365
   macro avg     0.7361    0.7685    0.7496       365
weighted avg     0.8438    0.8301    0.8356       365

AUROC (seed 0): 0.8737
AUPRC (seed 0): 0.5600
AUPRC class 0 (seed 0): 0.9689

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.64it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8682    0.9215    0.8940       293
           1     0.5741    0.4306    0.4921        72

    accuracy                         0.8247       365
   macro avg     0.7211    0.6760    0.6931       365
weighted avg     0.8102    0.8247    0.8147       365

AUROC (seed 1): 0.8621
AUPRC (seed 1): 0.5096
AUPRC class 0 (seed 1): 0.9658

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8647    0.8942    0.8792       293
           1     0.5000    0.4306    0.4627        72

    accuracy                         0.8027       365
   macro avg     0.6823    0.6624    0.6709       365
weighted avg     0.7927    0.8027    0.7970       365

AUROC (seed 2): 0.8042
AUPRC (seed 2): 0.4999
AUPRC class 0 (seed 2): 0.9411

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.54it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8635    0.9283    0.8947       293
           1     0.5800    0.4028    0.4754        72

    accuracy                         0.8247       365
   macro avg     0.7217    0.6656    0.6851       365
weighted avg     0.8076    0.8247    0.8120       365

AUROC (seed 3): 0.8702
AUPRC (seed 3): 0.5852
AUPRC class 0 (seed 3): 0.9667

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 37.63it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8851    0.8942    0.8896       293
           1     0.5507    0.5278    0.5390        72

    accuracy                         0.8219       365
   macro avg     0.7179    0.7110    0.7143       365
weighted avg     0.8192    0.8219    0.8205       365

AUROC (seed 4): 0.8810
AUPRC (seed 4): 0.6090
AUPRC class 0 (seed 4): 0.9701

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8737  AUPRC=0.5600AUPRC0=0.9689
Seed 1: AUROC=0.8621  AUPRC=0.5096AUPRC0=0.9658
Seed 2: AUROC=0.8042  AUPRC=0.4999AUPRC0=0.9411
Seed 3: AUROC=0.8702  AUPRC=0.5852AUPRC0=0.9667
Seed 4: AUROC=0.8810  AUPRC=0.6090AUPRC0=0.9701


In [24]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 25/25 [00:00<00:00, 25.10it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 36.93it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8527    0.9283    0.8889       293
           1     0.5435    0.3472    0.4237        72

    accuracy                         0.8137       365
   macro avg     0.6981    0.6378    0.6563       365
weighted avg     0.7917    0.8137    0.7971       365

AUROC (seed 0): 0.8815
AUPRC (seed 0): 0.5540
AUPRC class 0 (seed 0): 0.9702

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8567    0.8976    0.8767       293
           1     0.4828    0.3889    0.4308        72

    accuracy                         0.7973       365
   macro avg     0.6697    0.6432    0.6537       365
weighted avg     0.7829    0.7973    0.7887       365

AUROC (seed 3): 0.8593
AUPRC (seed 3): 0.5056
AUPRC class 0 (seed 3): 0.9647

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 25/25 [00:00<00:00, 27.99it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 25/25 [00:00<00:00, 36.62it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8607    0.9488    0.9026       293
           1     0.6429    0.3750    0.4737        72

    accuracy                         0.8356       365
   macro avg     0.7518    0.6619    0.6881       365
weighted avg     0.8177    0.8356    0.8180       365

AUROC (seed 4): 0.8833
AUPRC (seed 4): 0.5984
AUPRC class 0 (seed 4): 0.9707

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8815  AUPRC=0.5540AUPRC0=0.9702
Seed 1: AUROC=0.8776  AUPRC=0.5606AUPRC0=0.9688
Seed 2: AUROC=0.8863  AUPRC=0.5882AUPRC0=0.9719
Seed 3: AUROC=0.8593  AUPRC=0.5056AUPRC0=0.9647
Seed 4: AUROC=0.8833  AUPRC=0.5984AUPRC0=0.9707


In [25]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.04it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8485    0.9556    0.8989       293
           1     0.6286    0.3056    0.4112        72

    accuracy                         0.8274       365
   macro avg     0.7385    0.6306    0.6550       365
weighted avg     0.8051    0.8274    0.8027       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8757
AUPRC (MC mean over 100 samples, seed 0) : 0.5659
AUPRC class 0 (seed 0): 0.9687

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.17it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8483    0.9352    0.8896       293
           1     0.5476    0.3194    0.4035        72

    accuracy                         0.8137       365
   macro avg     0.6980    0.6273    0.6466       365
weighted avg     0.7890    0.8137    0.7937       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8771
AUPRC (MC mean over 100 samples, seed 1) : 0.5595
AUPRC class 0 (seed 1): 0.9697

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8754    0.9352    0.9043       293
           1     0.6346    0.4583    0.5323        72

    accuracy                         0.8411       365
   macro avg     0.7550    0.6967    0.7183       365
weighted avg     0.8279    0.8411    0.8309       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.8884
AUPRC (MC mean over 100 samples, seed 2) : 0.5989
AUPRC class 0 (seed 2): 0.9724

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.17it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8667    0.9317    0.8980       293
           1     0.6000    0.4167    0.4918        72

    accuracy                         0.8301       365
   macro avg     0.7333    0.6742    0.6949       365
weighted avg     0.8141    0.8301    0.8179       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.8807
AUPRC (MC mean over 100 samples, seed 3) : 0.5631
AUPRC class 0 (seed 3): 0.9707

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 25/25 [00:04<00:00,  5.17it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8746    0.9283    0.9007       293
           1     0.6111    0.4583    0.5238        72

    accuracy                         0.8356       365
   macro avg     0.7429    0.6933    0.7122       365
weighted avg     0.8226    0.8356    0.8263       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8767
AUPRC (MC mean over 100 samples, seed 4) : 0.5644
AUPRC class 0 (seed 4): 0.9698

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8757  AUPRC=0.5659AUPRC0=0.9687
Seed 1: AUROC=0.8771  AUPRC=0.5595AUPRC0=0.9697
Seed 2: AUROC=0.8884  AUPRC=0.5989AUPRC0=0.9724
Seed 3: AUROC=0.8807  AUPRC=0.56

In [14]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2019 or earlier for training and samples ending in 2020 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2019  
test_mask = forecast_end_years == 2020   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (13112, 30, 5) (13112, 7, 5) (13112,)
Test shapes: (366, 30, 5) (366, 7, 5) (366,)


In [15]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9046    0.9046    0.9046       325
           1     0.2439    0.2439    0.2439        41

    accuracy                         0.8306       366
   macro avg     0.5743    0.5743    0.5743       366
weighted avg     0.8306    0.8306    0.8306       366

AUROC (seed 0): 0.5743
AUPRC (seed 0): 0.1442
AUPRC class 0 (seed 0): 0.9030

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9147    0.8246    0.8673       325
           1     0.2192    0.3902    0.2807        41

    accuracy                         0.7760       366
   macro avg     0.5669    0.6074    0.5740       366
weighted avg     0.8368    0.7760    0.8016       366

AUROC (seed 1): 0.6074
AUPRC (seed 1): 0.1538
AUPRC class 0 (seed 1): 0.9100

Training MultiRocketHydraClassifi

In [27]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 32.23it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8988    0.9015    0.9002       325
           1     0.2000    0.1951    0.1975        41

    accuracy                         0.8224       366
   macro avg     0.5494    0.5483    0.5488       366
weighted avg     0.8205    0.8224    0.8214       366

AUROC (seed 0): 0.8232
AUPRC (seed 0): 0.2946
AUPRC class 0 (seed 0): 0.9767

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 37.83it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9114    0.8862    0.8986       325
           1     0.2600    0.3171    0.2857        41

    accuracy                         0.8224       366
   macro avg     0.5857    0.6016    0.5922       366
weighted avg     0.8384    0.8224    0.8299       366

AUROC (seed 1): 0.8261
AUPRC (seed 1): 0.3315
AUPRC class 0 (seed 1): 0.9764

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.9194    0.8769    0.8976       325
           1     0.2857    0.3902    0.3299        41

    accuracy                         0.8224       366
   macro avg     0.6025    0.6336    0.6138       366
weighted avg     0.8484    0.8224    0.8340       366

AUROC (seed 2): 0.8466
AUPRC (seed 2): 0.3211
AUPRC class 0 (seed 2): 0.9800

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 37.86it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9096    0.9292    0.9193       325
           1     0.3235    0.2683    0.2933        41

    accuracy                         0.8552       366
   macro avg     0.6166    0.5988    0.6063       366
weighted avg     0.8440    0.8552    0.8492       366

AUROC (seed 3): 0.8371
AUPRC (seed 3): 0.3249
AUPRC class 0 (seed 3): 0.9782

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 37.93it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9099    0.9015    0.9057       325
           1     0.2727    0.2927    0.2824        41

    accuracy                         0.8333       366
   macro avg     0.5913    0.5971    0.5940       366
weighted avg     0.8386    0.8333    0.8359       366

AUROC (seed 4): 0.8128
AUPRC (seed 4): 0.3016
AUPRC class 0 (seed 4): 0.9740

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8232  AUPRC=0.2946AUPRC0=0.9767
Seed 1: AUROC=0.8261  AUPRC=0.3315AUPRC0=0.9764
Seed 2: AUROC=0.8466  AUPRC=0.3211AUPRC0=0.9800
Seed 3: AUROC=0.8371  AUPRC=0.3249AUPRC0=0.9782
Seed 4: AUROC=0.8128  AUPRC=0.3016AUPRC0=0.9740


In [28]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 26/26 [00:00<00:00, 28.74it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 37.27it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9256    0.8800    0.9022       325
           1     0.3158    0.4390    0.3673        41

    accuracy                         0.8306       366
   macro avg     0.6207    0.6595    0.6348       366
weighted avg     0.8573    0.8306    0.8423       366

AUROC (seed 0): 0.8591
AUPRC (seed 0): 0.3658
AUPRC class 0 (seed 0): 0.9817

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9219    0.9446    0.9331       325
           1     0.4545    0.3659    0.4054        41

    accuracy                         0.8798       366
   macro avg     0.6882    0.6552    0.6693       366
weighted avg     0.8696    0.8798    0.8740       366

AUROC (seed 3): 0.7792
AUPRC (seed 3): 0.3407
AUPRC class 0 (seed 3): 0.9673

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 26/26 [00:00<00:00, 28.64it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 26/26 [00:00<00:00, 37.29it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9305    0.8646    0.8963       325
           1     0.3125    0.4878    0.3810        41

    accuracy                         0.8224       366
   macro avg     0.6215    0.6762    0.6386       366
weighted avg     0.8612    0.8224    0.8386       366

AUROC (seed 4): 0.8099
AUPRC (seed 4): 0.3216
AUPRC class 0 (seed 4): 0.9735

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8591  AUPRC=0.3658AUPRC0=0.9817
Seed 1: AUROC=0.8504  AUPRC=0.4550AUPRC0=0.9793
Seed 2: AUROC=0.8377  AUPRC=0.3051AUPRC0=0.9785
Seed 3: AUROC=0.7792  AUPRC=0.3407AUPRC0=0.9673
Seed 4: AUROC=0.8099  AUPRC=0.3216AUPRC0=0.9735


In [29]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 26/26 [00:04<00:00,  5.22it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.9387    0.9415    0.9401       325
           1     0.5250    0.5122    0.5185        41

    accuracy                         0.8934       366
   macro avg     0.7318    0.7269    0.7293       366
weighted avg     0.8923    0.8934    0.8929       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8718
AUPRC (MC mean over 100 samples, seed 0) : 0.5061
AUPRC class 0 (seed 0): 0.9821

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 26/26 [00:04<00:00,  5.22it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.9177    0.9262    0.9219       325
           1     0.3684    0.3415    0.3544        41

    accuracy                         0.8607       366
   macro avg     0.6431    0.6338    0.6382       366
weighted avg     0.8562    0.8607    0.8583       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8862
AUPRC (MC mean over 100 samples, seed 1) : 0.4070
AUPRC class 0 (seed 1): 0.9853

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9342    0.9169    0.9255       325
           1     0.4255    0.4878    0.4545        41

    accuracy                         0.8689       366
   macro avg     0.6799    0.7024    0.6900       366
weighted avg     0.8772    0.8689    0.8727       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.8704
AUPRC (MC mean over 100 samples, seed 2) : 0.4276
AUPRC class 0 (seed 2): 0.9823

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 26/26 [00:05<00:00,  5.11it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.9373    0.9200    0.9286       325
           1     0.4468    0.5122    0.4773        41

    accuracy                         0.8743       366
   macro avg     0.6921    0.7161    0.7029       366
weighted avg     0.8824    0.8743    0.8780       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.8582
AUPRC (MC mean over 100 samples, seed 3) : 0.5149
AUPRC class 0 (seed 3): 0.9792

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 26/26 [00:04<00:00,  5.22it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.9344    0.9200    0.9271       325
           1     0.4348    0.4878    0.4598        41

    accuracy                         0.8716       366
   macro avg     0.6846    0.7039    0.6935       366
weighted avg     0.8784    0.8716    0.8748       366

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8485
AUPRC (MC mean over 100 samples, seed 4) : 0.4993
AUPRC class 0 (seed 4): 0.9779

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8718  AUPRC=0.5061AUPRC0=0.9821
Seed 1: AUROC=0.8862  AUPRC=0.4070AUPRC0=0.9853
Seed 2: AUROC=0.8704  AUPRC=0.4276AUPRC0=0.9823
Seed 3: AUROC=0.8582  AUPRC=0.51

In [16]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2020 or earlier for training and samples ending in 2021 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2020  
test_mask = forecast_end_years == 2021   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (13478, 30, 5) (13478, 7, 5) (13478,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [17]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8696    0.8667    0.8681       300
           1     0.3939    0.4000    0.3969        65

    accuracy                         0.7836       365
   macro avg     0.6318    0.6333    0.6325       365
weighted avg     0.7849    0.7836    0.7842       365

AUROC (seed 0): 0.6333
AUPRC (seed 0): 0.2644
AUPRC class 0 (seed 0): 0.8632

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8333    0.8333    0.8333       300
           1     0.2308    0.2308    0.2308        65

    accuracy                         0.7260       365
   macro avg     0.5321    0.5321    0.5321       365
weighted avg     0.7260    0.7260    0.7260       365

AUROC (seed 1): 0.5321
AUPRC (seed 1): 0.1902
AUPRC class 0 (seed 1): 0.8314

Training MultiRocketHydraClassifi

In [31]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.63it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8933    0.8933    0.8933       300
           1     0.5077    0.5077    0.5077        65

    accuracy                         0.8247       365
   macro avg     0.7005    0.7005    0.7005       365
weighted avg     0.8247    0.8247    0.8247       365

AUROC (seed 0): 0.8521
AUPRC (seed 0): 0.5057
AUPRC class 0 (seed 0): 0.9665

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.87it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8896    0.9133    0.9013       300
           1     0.5439    0.4769    0.5082        65

    accuracy                         0.8356       365
   macro avg     0.7167    0.6951    0.7048       365
weighted avg     0.8280    0.8356    0.8313       365

AUROC (seed 1): 0.8537
AUPRC (seed 1): 0.5565
AUPRC class 0 (seed 1): 0.9627

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8746    0.9067    0.8903       300
           1     0.4815    0.4000    0.4370        65

    accuracy                         0.8164       365
   macro avg     0.6780    0.6533    0.6637       365
weighted avg     0.8046    0.8164    0.8096       365

AUROC (seed 2): 0.8658
AUPRC (seed 2): 0.5714
AUPRC class 0 (seed 2): 0.9694

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.83it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8721    0.8633    0.8677       300
           1     0.3971    0.4154    0.4060        65

    accuracy                         0.7836       365
   macro avg     0.6346    0.6394    0.6368       365
weighted avg     0.7875    0.7836    0.7855       365

AUROC (seed 3): 0.8209
AUPRC (seed 3): 0.4926
AUPRC class 0 (seed 3): 0.9581

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.70it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8602    0.9433    0.8998       300
           1     0.5278    0.2923    0.3762        65

    accuracy                         0.8274       365
   macro avg     0.6940    0.6178    0.6380       365
weighted avg     0.8010    0.8274    0.8066       365

AUROC (seed 4): 0.8444
AUPRC (seed 4): 0.5132
AUPRC class 0 (seed 4): 0.9639

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8521  AUPRC=0.5057AUPRC0=0.9665
Seed 1: AUROC=0.8537  AUPRC=0.5565AUPRC0=0.9627
Seed 2: AUROC=0.8658  AUPRC=0.5714AUPRC0=0.9694
Seed 3: AUROC=0.8209  AUPRC=0.4926AUPRC0=0.9581
Seed 4: AUROC=0.8444  AUPRC=0.5132AUPRC0=0.9639


In [32]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 27/27 [00:00<00:00, 28.77it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.18it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8730    0.9167    0.8943       300
           1     0.5000    0.3846    0.4348        65

    accuracy                         0.8219       365
   macro avg     0.6865    0.6506    0.6645       365
weighted avg     0.8066    0.8219    0.8125       365

AUROC (seed 0): 0.8614
AUPRC (seed 0): 0.5821
AUPRC class 0 (seed 0): 0.9685

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9033    0.9033    0.9033       300
           1     0.5538    0.5538    0.5538        65

    accuracy                         0.8411       365
   macro avg     0.7286    0.7286    0.7286       365
weighted avg     0.8411    0.8411    0.8411       365

AUROC (seed 3): 0.8786
AUPRC (seed 3): 0.6385
AUPRC class 0 (seed 3): 0.9718

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 27/27 [00:00<00:00, 28.81it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 27/27 [00:00<00:00, 37.31it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8835    0.9100    0.8966       300
           1     0.5179    0.4462    0.4793        65

    accuracy                         0.8274       365
   macro avg     0.7007    0.6781    0.6879       365
weighted avg     0.8184    0.8274    0.8223       365

AUROC (seed 4): 0.8488
AUPRC (seed 4): 0.5778
AUPRC class 0 (seed 4): 0.9622

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8614  AUPRC=0.5821AUPRC0=0.9685
Seed 1: AUROC=0.8272  AUPRC=0.4683AUPRC0=0.9611
Seed 2: AUROC=0.8502  AUPRC=0.5873AUPRC0=0.9629
Seed 3: AUROC=0.8786  AUPRC=0.6385AUPRC0=0.9718
Seed 4: AUROC=0.8488  AUPRC=0.5778AUPRC0=0.9622


In [33]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 27/27 [00:05<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8660    0.9267    0.8953       300
           1     0.5000    0.3385    0.4037        65

    accuracy                         0.8219       365
   macro avg     0.6830    0.6326    0.6495       365
weighted avg     0.8009    0.8219    0.8078       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8318
AUPRC (MC mean over 100 samples, seed 0) : 0.5517
AUPRC class 0 (seed 0): 0.9569

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 27/27 [00:05<00:00,  5.23it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8779    0.8867    0.8823       300
           1     0.4516    0.4308    0.4409        65

    accuracy                         0.8055       365
   macro avg     0.6648    0.6587    0.6616       365
weighted avg     0.8020    0.8055    0.8037       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8413
AUPRC (MC mean over 100 samples, seed 1) : 0.5520
AUPRC class 0 (seed 1): 0.9621

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8675    0.9167    0.8914       300
           1     0.4792    0.3538    0.4071        65

    accuracy                         0.8164       365
   macro avg     0.6733    0.6353    0.6492       365
weighted avg     0.7984    0.8164    0.8052       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.8667
AUPRC (MC mean over 100 samples, seed 2) : 0.5986
AUPRC class 0 (seed 2): 0.9688

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 27/27 [00:05<00:00,  5.25it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8758    0.9167    0.8958       300
           1     0.5098    0.4000    0.4483        65

    accuracy                         0.8247       365
   macro avg     0.6928    0.6583    0.6720       365
weighted avg     0.8106    0.8247    0.8161       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.8702
AUPRC (MC mean over 100 samples, seed 3) : 0.6008
AUPRC class 0 (seed 3): 0.9702

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 27/27 [00:05<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8778    0.9100    0.8936       300
           1     0.5000    0.4154    0.4538        65

    accuracy                         0.8219       365
   macro avg     0.6889    0.6627    0.6737       365
weighted avg     0.8105    0.8219    0.8153       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8474
AUPRC (MC mean over 100 samples, seed 4) : 0.5662
AUPRC class 0 (seed 4): 0.9629

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8318  AUPRC=0.5517AUPRC0=0.9569
Seed 1: AUROC=0.8413  AUPRC=0.5520AUPRC0=0.9621
Seed 2: AUROC=0.8667  AUPRC=0.5986AUPRC0=0.9688
Seed 3: AUROC=0.8702  AUPRC=0.60

In [18]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2021 or earlier for training and samples ending in 2022 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2021  
test_mask = forecast_end_years == 2022   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (13843, 30, 5) (13843, 7, 5) (13843,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [19]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.7024    0.8712    0.7778       233
           1     0.6053    0.3485    0.4423       132

    accuracy                         0.6822       365
   macro avg     0.6538    0.6099    0.6100       365
weighted avg     0.6673    0.6822    0.6565       365

AUROC (seed 0): 0.6099
AUPRC (seed 0): 0.4465
AUPRC class 0 (seed 0): 0.6942

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.7168    0.8584    0.7812       233
           1     0.6163    0.4015    0.4862       132

    accuracy                         0.6932       365
   macro avg     0.6666    0.6299    0.6337       365
weighted avg     0.6805    0.6932    0.6746       365

AUROC (seed 1): 0.6299
AUPRC (seed 1): 0.4639
AUPRC class 0 (seed 1): 0.7057

Training MultiRocketHydraClassifi

In [35]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 32.87it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8667    0.8927    0.8795       233
           1     0.8000    0.7576    0.7782       132

    accuracy                         0.8438       365
   macro avg     0.8333    0.8251    0.8289       365
weighted avg     0.8426    0.8438    0.8429       365

AUROC (seed 0): 0.9287
AUPRC (seed 0): 0.8867
AUPRC class 0 (seed 0): 0.9583

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.83it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8480    0.9099    0.8778       233
           1     0.8174    0.7121    0.7611       132

    accuracy                         0.8384       365
   macro avg     0.8327    0.8110    0.8195       365
weighted avg     0.8369    0.8384    0.8356       365

AUROC (seed 1): 0.9358
AUPRC (seed 1): 0.8962
AUPRC class 0 (seed 1): 0.9648

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8520    0.9142    0.8820       233
           1     0.8261    0.7197    0.7692       132

    accuracy                         0.8438       365
   macro avg     0.8390    0.8169    0.8256       365
weighted avg     0.8426    0.8438    0.8412       365

AUROC (seed 2): 0.9210
AUPRC (seed 2): 0.8696
AUPRC class 0 (seed 2): 0.9561

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.85it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8755    0.8755    0.8755       233
           1     0.7803    0.7803    0.7803       132

    accuracy                         0.8411       365
   macro avg     0.8279    0.8279    0.8279       365
weighted avg     0.8411    0.8411    0.8411       365

AUROC (seed 3): 0.9267
AUPRC (seed 3): 0.8813
AUPRC class 0 (seed 3): 0.9595

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.88it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8554    0.8884    0.8716       233
           1     0.7886    0.7348    0.7608       132

    accuracy                         0.8329       365
   macro avg     0.8220    0.8116    0.8162       365
weighted avg     0.8312    0.8329    0.8315       365

AUROC (seed 4): 0.9216
AUPRC (seed 4): 0.8663
AUPRC class 0 (seed 4): 0.9565

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9287  AUPRC=0.8867AUPRC0=0.9583
Seed 1: AUROC=0.9358  AUPRC=0.8962AUPRC0=0.9648
Seed 2: AUROC=0.9210  AUPRC=0.8696AUPRC0=0.9561
Seed 3: AUROC=0.9267  AUPRC=0.8813AUPRC0=0.9595
Seed 4: AUROC=0.9216  AUPRC=0.8663AUPRC0=0.9565


In [36]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 28/28 [00:00<00:00, 28.82it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.30it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8116    0.9614    0.8802       233
           1     0.8989    0.6061    0.7240       132

    accuracy                         0.8329       365
   macro avg     0.8552    0.7837    0.8021       365
weighted avg     0.8432    0.8329    0.8237       365

AUROC (seed 0): 0.9342
AUPRC (seed 0): 0.8943
AUPRC class 0 (seed 0): 0.9644

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8935    0.8283    0.8597       233
           1     0.7315    0.8258    0.7758       132

    accuracy                         0.8274       365
   macro avg     0.8125    0.8270    0.8177       365
weighted avg     0.8349    0.8274    0.8294       365

AUROC (seed 3): 0.9050
AUPRC (seed 3): 0.8491
AUPRC class 0 (seed 3): 0.9464

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 28/28 [00:00<00:00, 28.37it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 31.63it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8413    0.9099    0.8742       233
           1     0.8142    0.6970    0.7510       132

    accuracy                         0.8329       365
   macro avg     0.8277    0.8034    0.8126       365
weighted avg     0.8315    0.8329    0.8297       365

AUROC (seed 4): 0.9266
AUPRC (seed 4): 0.8847
AUPRC class 0 (seed 4): 0.9599

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9342  AUPRC=0.8943AUPRC0=0.9644
Seed 1: AUROC=0.9158  AUPRC=0.8605AUPRC0=0.9547
Seed 2: AUROC=0.9311  AUPRC=0.8850AUPRC0=0.9625
Seed 3: AUROC=0.9050  AUPRC=0.8491AUPRC0=0.9464
Seed 4: AUROC=0.9266  AUPRC=0.8847AUPRC0=0.9599


In [37]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8429    0.9442    0.8907       233
           1     0.8750    0.6894    0.7712       132

    accuracy                         0.8521       365
   macro avg     0.8590    0.8168    0.8309       365
weighted avg     0.8545    0.8521    0.8475       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.9368
AUPRC (MC mean over 100 samples, seed 0) : 0.9035
AUPRC class 0 (seed 0): 0.9634

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8730    0.9142    0.8931       233
           1     0.8347    0.7652    0.7984       132

    accuracy                         0.8603       365
   macro avg     0.8538    0.8397    0.8458       365
weighted avg     0.8591    0.8603    0.8588       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9371
AUPRC (MC mean over 100 samples, seed 1) : 0.9015
AUPRC class 0 (seed 1): 0.9644

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8450    0.9356    0.8880       233
           1     0.8598    0.6970    0.7699       132

    accuracy                         0.8493       365
   macro avg     0.8524    0.8163    0.8289       365
weighted avg     0.8503    0.8493    0.8453       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9258
AUPRC (MC mean over 100 samples, seed 2) : 0.8863
AUPRC class 0 (seed 2): 0.9533

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.27it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.9330    0.8369    0.8824       233
           1     0.7564    0.8939    0.8194       132

    accuracy                         0.8575       365
   macro avg     0.8447    0.8654    0.8509       365
weighted avg     0.8691    0.8575    0.8596       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9342
AUPRC (MC mean over 100 samples, seed 3) : 0.9050
AUPRC class 0 (seed 3): 0.9580

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.26it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8155    0.9485    0.8770       233
           1     0.8723    0.6212    0.7257       132

    accuracy                         0.8301       365
   macro avg     0.8439    0.7849    0.8013       365
weighted avg     0.8361    0.8301    0.8223       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9301
AUPRC (MC mean over 100 samples, seed 4) : 0.8883
AUPRC class 0 (seed 4): 0.9590

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.9368  AUPRC=0.9035AUPRC0=0.9634
Seed 1: AUROC=0.9371  AUPRC=0.9015AUPRC0=0.9644
Seed 2: AUROC=0.9258  AUPRC=0.8863AUPRC0=0.9533
Seed 3: AUROC=0.9342  AUPRC=0.90

In [20]:
"""
This code loads and prepares multiple Shanghai weather CSV files.

It first finds all Shanghai weather files in the specified folder, sorts them,
and reads each file into a DataFrame. Because the files contain header metadata
before the actual table, the code searches for the line beginning with "YEAR"
and skips all earlier lines when loading the CSV. The year is extracted from
each filename and stored as an additional column before all yearly files are
combined into one DataFrame.

The code then selects five weather-related features and creates sliding-window
samples. Each sample contains:
- 30 previous days of observed weather data,
- 7 future days of weather data used to define the target label.

Hot and cold extreme thresholds are calculated using the 95th percentile of
T2M_MAX and the 5th percentile of T2M_MIN. A binary label is assigned to each
sample: class 1 if the future 7-day window contains either an unusually hot day
or an unusually cold day, and class 0 otherwise.

The observed windows, forecast windows, and labels are converted into NumPy
arrays. The data is then split chronologically by year, using samples ending in
2022 or earlier for training and samples ending in 2023 for testing.

Finally, the code prints the shapes of the training and testing arrays so the
user can verify that the weather dataset has been prepared correctly.
"""
import pandas as pd
import glob
import os
from tensorflow.keras.layers import Reshape
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
folder_path = "/Users/user/Desktop/Project 2 examples/weather_data"
csv_files = sorted(glob.glob(os.path.join(folder_path, "Shanghai_*.csv")))
df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    year = int(filename.split('_')[1].split('.')[0])
    with open(file, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
    df = pd.read_csv(file, skiprows=start_idx)
    df["year_file"] = year
    df_list.append(df)
combined_shanghai = pd.concat(df_list, ignore_index=True)
combined_df = combined_shanghai
features = ['WS10M', 'RH2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
lookback = 30
forecast_horizon = 7
data = combined_df[features].values
hot_threshold = np.percentile(combined_df['T2M_MAX'], 95)
cold_threshold = np.percentile(combined_df['T2M_MIN'], 5)
X_observed, X_forecast, y = [], [], []
for i in range(len(data) - lookback - forecast_horizon):
    X_observed.append(data[i : i + lookback])
    X_forecast.append(data[i + lookback : i + lookback + forecast_horizon])
    future_window = data[i + lookback : i + lookback + forecast_horizon]
    future_tmax = future_window[:, features.index('T2M_MAX')]
    future_tmin = future_window[:, features.index('T2M_MIN')]
    label = 1 if (np.max(future_tmax) > hot_threshold) or (np.min(future_tmin) < cold_threshold) else 0
    y.append(label)
X_observed = np.array(X_observed)
X_forecast = np.array(X_forecast)
y = np.array(y)
forecast_end_years = combined_df['YEAR'].values[lookback + forecast_horizon : len(data)]
train_mask = forecast_end_years <= 2022  
test_mask = forecast_end_years == 2023   
X_observed_train = X_observed[train_mask]
X_forecast_train = X_forecast[train_mask]
y_train = y[train_mask]
X_observed_test = X_observed[test_mask]
X_forecast_test = X_forecast[test_mask]
y_test = y[test_mask]
print("Training shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes:", X_observed_test.shape, X_forecast_test.shape, y_test.shape)

Training shapes: (14208, 30, 5) (14208, 7, 5) (14208,)
Test shapes: (365, 30, 5) (365, 7, 5) (365,)


In [21]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )



Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.7984    0.7384    0.7672       279
           1     0.3178    0.3953    0.3523        86

    accuracy                         0.6575       365
   macro avg     0.5581    0.5669    0.5598       365
weighted avg     0.6852    0.6575    0.6695       365

AUROC (seed 0): 0.5669
AUPRC (seed 0): 0.2681
AUPRC class 0 (seed 0): 0.7895

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.7815    0.8459    0.8124       279
           1     0.3175    0.2326    0.2685        86

    accuracy                         0.7014       365
   macro avg     0.5495    0.5392    0.5404       365
weighted avg     0.6721    0.7014    0.6842       365

AUROC (seed 1): 0.5392
AUPRC (seed 1): 0.2546
AUPRC class 0 (seed 1): 0.7788

Training MultiRocketHydraClassifi

In [39]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.77it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8723    0.8566    0.8644       279
           1     0.5604    0.5930    0.5763        86

    accuracy                         0.7945       365
   macro avg     0.7164    0.7248    0.7203       365
weighted avg     0.7988    0.7945    0.7965       365

AUROC (seed 0): 0.8277
AUPRC (seed 0): 0.6077
AUPRC class 0 (seed 0): 0.9409

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.70it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8459    0.8459    0.8459       279
           1     0.5000    0.5000    0.5000        86

    accuracy                         0.7644       365
   macro avg     0.6729    0.6729    0.6729       365
weighted avg     0.7644    0.7644    0.7644       365

AUROC (seed 1): 0.7998
AUPRC (seed 1): 0.5468
AUPRC class 0 (seed 1): 0.9317

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8387    0.8387    0.8387       279
           1     0.4767    0.4767    0.4767        86

    accuracy                         0.7534       365
   macro avg     0.6577    0.6577    0.6577       365
weighted avg     0.7534    0.7534    0.7534       365

AUROC (seed 2): 0.7815
AUPRC (seed 2): 0.5059
AUPRC class 0 (seed 2): 0.9146

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.85it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8114    0.8638    0.8368       279
           1     0.4412    0.3488    0.3896        86

    accuracy                         0.7425       365
   macro avg     0.6263    0.6063    0.6132       365
weighted avg     0.7242    0.7425    0.7314       365

AUROC (seed 3): 0.7697
AUPRC (seed 3): 0.5242
AUPRC class 0 (seed 3): 0.9189

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.76it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8495    0.8495    0.8495       279
           1     0.5116    0.5116    0.5116        86

    accuracy                         0.7699       365
   macro avg     0.6805    0.6805    0.6805       365
weighted avg     0.7699    0.7699    0.7699       365

AUROC (seed 4): 0.8065
AUPRC (seed 4): 0.6012
AUPRC class 0 (seed 4): 0.9274

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.8277  AUPRC=0.6077AUPRC0=0.9409
Seed 1: AUROC=0.7998  AUPRC=0.5468AUPRC0=0.9317
Seed 2: AUROC=0.7815  AUPRC=0.5059AUPRC0=0.9146
Seed 3: AUROC=0.7697  AUPRC=0.5242AUPRC0=0.9189
Seed 4: AUROC=0.8065  AUPRC=0.6012AUPRC0=0.9274


In [40]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 28/28 [00:00<00:00, 28.60it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 37.04it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8288    0.8674    0.8476       279
           1     0.4932    0.4186    0.4528        86

    accuracy                         0.7616       365
   macro avg     0.6610    0.6430    0.6502       365
weighted avg     0.7497    0.7616    0.7546       365

AUROC (seed 0): 0.7750
AUPRC (seed 0): 0.5206
AUPRC class 0 (seed 0): 0.9156

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8538    0.7957    0.8237       279
           1     0.4571    0.5581    0.5026        86

    accuracy                         0.7397       365
   macro avg     0.6555    0.6769    0.6632       365
weighted avg     0.7604    0.7397    0.7481       365

AUROC (seed 3): 0.7814
AUPRC (seed 3): 0.5202
AUPRC class 0 (seed 3): 0.9252

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 28/28 [00:00<00:00, 28.56it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 28/28 [00:00<00:00, 36.97it/s]


Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8797    0.7599    0.8154       279
           1     0.4597    0.6628    0.5429        86

    accuracy                         0.7370       365
   macro avg     0.6697    0.7113    0.6791       365
weighted avg     0.7807    0.7370    0.7512       365

AUROC (seed 4): 0.8038
AUPRC (seed 4): 0.5361
AUPRC class 0 (seed 4): 0.9336

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.7750  AUPRC=0.5206AUPRC0=0.9156
Seed 1: AUROC=0.7921  AUPRC=0.5768AUPRC0=0.9220
Seed 2: AUROC=0.7663  AUPRC=0.5016AUPRC0=0.9191
Seed 3: AUROC=0.7814  AUPRC=0.5202AUPRC0=0.9252
Seed 4: AUROC=0.8038  AUPRC=0.5361AUPRC0=0.9336


In [41]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.19it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8135    0.9068    0.8576       279
           1     0.5185    0.3256    0.4000        86

    accuracy                         0.7699       365
   macro avg     0.6660    0.6162    0.6288       365
weighted avg     0.7440    0.7699    0.7498       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.8030
AUPRC (MC mean over 100 samples, seed 0) : 0.5859
AUPRC class 0 (seed 0): 0.9292

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.19it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8551    0.8674    0.8612       279
           1     0.5488    0.5233    0.5357        86

    accuracy                         0.7863       365
   macro avg     0.7020    0.6953    0.6985       365
weighted avg     0.7829    0.7863    0.7845       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.8035
AUPRC (MC mean over 100 samples, seed 1) : 0.5907
AUPRC class 0 (seed 1): 0.9295

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8322    0.8710    0.8511       279
           1     0.5068    0.4302    0.4654        86

    accuracy                         0.7671       365
   macro avg     0.6695    0.6506    0.6583       365
weighted avg     0.7555    0.7671    0.7603       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.7917
AUPRC (MC mean over 100 samples, seed 2) : 0.5709
AUPRC class 0 (seed 2): 0.9262

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.20it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8072    0.8853    0.8444       279
           1     0.4576    0.3140    0.3724        86

    accuracy                         0.7507       365
   macro avg     0.6324    0.5996    0.6084       365
weighted avg     0.7248    0.7507    0.7332       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.7881
AUPRC (MC mean over 100 samples, seed 3) : 0.5464
AUPRC class 0 (seed 3): 0.9253

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 28/28 [00:05<00:00,  5.20it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8480    0.8996    0.8730       279
           1     0.5942    0.4767    0.5290        86

    accuracy                         0.8000       365
   macro avg     0.7211    0.6882    0.7010       365
weighted avg     0.7882    0.8000    0.7920       365

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.8057
AUPRC (MC mean over 100 samples, seed 4) : 0.5977
AUPRC class 0 (seed 4): 0.9312

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.8030  AUPRC=0.5859AUPRC0=0.9292
Seed 1: AUROC=0.8035  AUPRC=0.5907AUPRC0=0.9295
Seed 2: AUROC=0.7917  AUPRC=0.5709AUPRC0=0.9262
Seed 3: AUROC=0.7881  AUPRC=0.54